# Классификация — KNN, Naive Bayes, Decision Tree, Logistic Regression

Подбор гиперпараметров через **Optuna**.

## 0. Импорты

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set(style='whitegrid')
%matplotlib inline

## 1. Загрузка данных

👉 **Подставь свой датасет и имя таргета.**

In [ ]:
# ===== ТВОИ НАСТРОЙКИ =====
DATASET_PATH = 'your_dataset.csv'   # путь к файлу
TARGET_COL   = 'target'             # имя целевого столбца
# ===========================

df = pd.read_csv(DATASET_PATH)
print(f'Shape: {df.shape}')
df.head()

## 2. Предобработка

In [ ]:
# Удаляем строки с пропусками
df = df.dropna()
print(f'After dropna: {df.shape}')

# Кодируем таргет, если он строковый
le = LabelEncoder()
y = le.fit_transform(df[TARGET_COL])
X = df.drop(columns=[TARGET_COL])

# Оставляем только числовые признаки
X = X.select_dtypes(include=[np.number])
print(f'Features: {X.shape[1]}, Classes: {len(le.classes_)}')

## 3. Исследовательский анализ (EDA)

👉 **Укажи список столбцов для отрисовки в `PLOT_COLS`.**

In [ ]:
# ===== СТОЛБЦЫ ДЛЯ ГРАФИКОВ =====
PLOT_COLS = list(X.columns)[:6]   # первые 6 признаков (измени по желанию)
# =================================

fig, axes = plt.subplots(1, len(PLOT_COLS), figsize=(5 * len(PLOT_COLS), 4))
if len(PLOT_COLS) == 1:
    axes = [axes]
for ax, col in zip(axes, PLOT_COLS):
    ax.hist(X[col], bins=30, edgecolor='black')
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot: каждый выбранный признак по классам
plot_df = X[PLOT_COLS].copy()
plot_df['class'] = le.inverse_transform(y)

fig, axes = plt.subplots(1, len(PLOT_COLS), figsize=(5 * len(PLOT_COLS), 4))
if len(PLOT_COLS) == 1:
    axes = [axes]
for ax, col in zip(axes, PLOT_COLS):
    plot_df.boxplot(column=col, by='class', ax=ax)
    ax.set_title(col)
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Распределение классов
class_counts = pd.Series(y).value_counts().sort_index()
class_counts.index = le.inverse_transform(class_counts.index)

plt.figure(figsize=(6, 4))
class_counts.plot.bar(edgecolor='black')
plt.title('Class distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Корреляционная матрица
corr = pd.concat([X[PLOT_COLS], pd.Series(y, name='target')], axis=1).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

## 4. Train/Test split + масштабирование

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}')

## 5. Optuna — подбор гиперпараметров

In [ ]:
N_TRIALS = 50   # количество итераций Optuna

### 5.1 KNN

In [ ]:
def objective_knn(trial):
    n_neighbors = trial.suggest_int('n_neighbors', 1, 50)
    weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
    p = trial.suggest_int('p', 1, 2)  # 1=Manhattan, 2=Euclidean
    model = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights, p=p)
    score = cross_val_score(model, X_train_scaled, y_train,
                            cv=5, scoring='accuracy').mean()
    return score

study_knn = optuna.create_study(direction='maximize')
study_knn.optimize(objective_knn, n_trials=N_TRIALS)
print('KNN best params:', study_knn.best_params)

### 5.2 Naive Bayes

In [ ]:
def objective_nb(trial):
    var_smoothing = trial.suggest_float('var_smoothing', 1e-11, 1e-1, log=True)
    model = GaussianNB(var_smoothing=var_smoothing)
    score = cross_val_score(model, X_train_scaled, y_train,
                            cv=5, scoring='accuracy').mean()
    return score

study_nb = optuna.create_study(direction='maximize')
study_nb.optimize(objective_nb, n_trials=N_TRIALS)
print('NB best params:', study_nb.best_params)

### 5.3 Decision Tree (классификация)

In [ ]:
def objective_dt(trial):
    max_depth = trial.suggest_int('max_depth', 2, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 20)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        criterion=criterion,
        random_state=42
    )
    score = cross_val_score(model, X_train, y_train,
                            cv=5, scoring='accuracy').mean()
    return score

study_dt = optuna.create_study(direction='maximize')
study_dt.optimize(objective_dt, n_trials=N_TRIALS)
print('DT best params:', study_dt.best_params)

### 5.4 Logistic Regression (классификация)

In [ ]:
def objective_lr(trial):
    C = trial.suggest_float('C', 1e-3, 100.0, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])
    solver = 'liblinear'  # поддерживает l1 и l2
    model = LogisticRegression(C=C, penalty=penalty, solver=solver,
                               max_iter=1000, random_state=42)
    score = cross_val_score(model, X_train_scaled, y_train,
                            cv=5, scoring='accuracy').mean()
    return score

study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_lr, n_trials=N_TRIALS)
print('LR best params:', study_lr.best_params)

## 6. Обучение финальных моделей и метрики

In [ ]:
models = {
    'KNN': KNeighborsClassifier(**study_knn.best_params),
    'NaiveBayes': GaussianNB(**study_nb.best_params),
    'DecisionTree': DecisionTreeClassifier(**study_dt.best_params, random_state=42),
    'LogisticRegression': LogisticRegression(
        **study_lr.best_params, solver='liblinear', max_iter=1000, random_state=42
    ),
}

results = []

for name, model in models.items():
    # KNN, NB, LR — масштабированные; DT — оригинальные
    if name in ('KNN', 'NaiveBayes', 'LogisticRegression'):
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1})

results_df = pd.DataFrame(results).sort_values('F1', ascending=False)
results_df

## 7. Визуализация сравнения моделей

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, metric in zip(axes, ['Accuracy', 'Precision', 'Recall', 'F1']):
    bars = ax.bar(results_df['Model'], results_df[metric], edgecolor='black')
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=30)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h, f'{h:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 8. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4))
if len(models) == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, models.items()):
    if name in ('KNN', 'NaiveBayes', 'LogisticRegression'):
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()

## 9. Classification report (лучшая модель)

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = models[best_name]

if best_name in ('KNN', 'NaiveBayes', 'LogisticRegression'):
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

print(f'Лучшая модель: {best_name}')
print(classification_report(y_test, y_pred_best, target_names=le.classes_.astype(str)))

## 10. Важность признаков (Decision Tree)

In [ ]:
dt_model = models['DecisionTree']
feat_imp = pd.Series(dt_model.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=True).tail(15)

plt.figure(figsize=(6, max(3, len(feat_imp) * 0.35)))
feat_imp.plot.barh(edgecolor='black')
plt.title('Decision Tree Feature Importances')
plt.tight_layout()
plt.show()

## 11. Деплой лучшей модели

In [ ]:
import joblib

# Сохраняем модель, скейлер и LabelEncoder
joblib.dump(best_model, 'best_classification_model.pkl')
if best_name in ('KNN', 'NaiveBayes', 'LogisticRegression'):
    joblib.dump(scaler, 'classification_scaler.pkl')
joblib.dump(le, 'classification_label_encoder.pkl')

print(f'Лучшая модель: {best_name}')
print(f'F1 = {results_df.iloc[0]["F1"]:.4f}')
print('Сохранено в best_classification_model.pkl')